<a href="https://colab.research.google.com/github/fathinahnj/ggf-guava-spots-project/blob/main/detect_guava_spots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install -q ultralytics opencv-python-headless ipywidgets

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="SlBxRabVfF2QYLeZwq44")
project = rf.workspace("fathinahs-workspace").project("guava-seg-hwznh")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


In [38]:
from google.colab import files
import os
import zipfile
import glob

BASE_DIR = "/content/drive/MyDrive/ggf_guava_project"

FRAME_DIR = project.search(
    limit=10,
    in_dataset=True,
)

SEG_MODEL_PATH = os.path.join(BASE_DIR, 'weights.pt')
TRACK_MODEL_PATH = os.path.join(BASE_DIR, 'tracking_guava_seg_model.pt')

TRACKER_CONFIG = "bytetrack.yaml"

TRACK_CLASS_ID = None

OUTPUT_DIR = "/content/drive/MyDrive/ggf_guava_project/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [26]:
segmentation_model = YOLO(SEG_MODEL_PATH)
tracking_model = YOLO(TRACK_MODEL_PATH)

In [27]:
search_path = os.path.join(dataset.location, '**/*.jpg')
image_paths = glob.glob(search_path, recursive=True)

# Urutkan berdasarkan nama file
image_paths = sorted(image_paths)[:10]

if len(image_paths) == 0:
    raise FileNotFoundError(f"Tidak ada gambar yang ditemukan di {dataset.location}.")

print(f"Jumlah frame yang akan diproses: {len(image_paths)}")

for path in image_paths:
    print(os.path.basename(path))

Jumlah frame yang akan diproses: 10
20261117_20261117115107_20261117115714_115106_mp4-0009_jpg.rf.b7428bc48d247c9da1e4036de53df9c3.jpg
20261117_20261117115107_20261117115714_115106_mp4-0040_jpg.rf.0704f8f0a6c9be1b02eeb15c1a131b6a.jpg
20261117_20261117115107_20261117115714_115106_mp4-0047_jpg.rf.f1d1cac4be03925734f76ba1e8c1b8f8.jpg
20261117_20261117115107_20261117115714_115106_mp4-0054_jpg.rf.7f9b044540c18a52a419e8470fc74d3d.jpg
20261117_20261117115107_20261117115714_115106_mp4-0073_jpg.rf.d4b3cd843fc65f3d17962ba39805260c.jpg
20261117_20261117115107_20261117115714_115106_mp4-0088_jpg.rf.13e2c7ae8c3f1c2ea2af5efff5053254.jpg
20261117_20261117115107_20261117115714_115106_mp4-0101_jpg.rf.e65dc41ffb99b0cf7a22f48d92278900.jpg
20261117_20261117115107_20261117115714_115106_mp4-0118_jpg.rf.40f66a7bde7ad1c76612446a4c7bf86f.jpg
20261117_20261117115107_20261117115714_115106_mp4-0124_jpg.rf.6ba3ca9fc8d27f3427535973055facb9.jpg
20261117_20261117115107_20261117115714_115106_mp4-0127_jpg.rf.5b5581654bb

In [29]:
# Pengaturan threshold

import ipywidgets as widgets
from IPython.display import display

h_max_slider = widgets.IntSlider(
    value=25, min=0, max=179,
    description="H max:",
    continuous_update=False
)

s_min_slider = widgets.IntSlider(
    value=0, min=0, max=255,
    description="S min:",
    continuous_update=False
)

v_max_slider = widgets.IntSlider(
    value=190, min=0, max=255,
    description="V max:",
    continuous_update=False
)

conf_slider = widgets.FloatSlider(
    value=0.50, min=0.10, max=0.95, step=0.05,
    description="Confidence:",
    continuous_update=False
)

display(h_max_slider, s_min_slider, v_max_slider, conf_slider)

IntSlider(value=25, continuous_update=False, description='H max:', max=179)

IntSlider(value=0, continuous_update=False, description='S min:', max=255)

IntSlider(value=190, continuous_update=False, description='V max:', max=255)

FloatSlider(value=0.5, continuous_update=False, description='Confidence:', max=0.95, min=0.1, step=0.05)

In [32]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simpan hasil segmentasi buah agar tidak perlu inference ulang
frame_paths = []

for image_path in image_paths:

    image = cv2.imread(image_path)

    if image is None:
        print(f"Gagal membaca: {image_path}")
        continue

    height, width = image.shape[:2]

    # Inference model segmentasi buah
    results = model.predict(
        source=image,
        conf=conf_slider.value,
        retina_masks=True,
        verbose=False
    )

    result = results[0]

    frame_paths.append({
        "path": image_path,
        "image": image,
        "fruit_mask": fruit_mask
    })

print(f"Segmentasi selesai: {len(frame_paths)} frame.")

Segmentasi selesai: 10 frame.


In [35]:
results_table = []
visualization_data = []

# Definisikan parameter confidence tracking secara default atau gunakan slider
TRACK_CONF = conf_slider.value
SEG_CONF = conf_slider.value

# Helper function untuk deteksi bercak jika belum didefinisikan
if 'detect_spots' not in globals():
    def detect_spots(image, fruit_mask):
        # Konversi ke HSV
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        # Ambil nilai dari slider
        lower = np.array([0, s_min_slider.value, 0])
        upper = np.array([h_max_slider.value, 255, v_max_slider.value])
        spot_mask = cv2.inRange(hsv, lower, upper)
        # Batasi hanya pada area buah
        return cv2.bitwise_and(spot_mask, fruit_mask)

for frame_index, frame_data_item in enumerate(frame_paths):

    print(f"Processing frame {frame_index + 1}/{len(frame_paths)}")

    # Ambil data path dan image langsung dari dictionary hasil segmentasi sebelumnya
    frame_path_str = frame_data_item["path"]
    image = frame_data_item["image"]

    if image is None:
        continue

    height, width = image.shape[:2]

    # ==========================================
    # A. TRACKING
    # ==========================================

    tracking_results = tracking_model.track(
        source=image,
        persist=True,
        tracker=TRACKER_CONFIG,
        conf=TRACK_CONF,
        classes=[TRACK_CLASS_ID] if TRACK_CLASS_ID is not None else None,
        verbose=False
    )

    tracking_result = tracking_results[0]

    if tracking_result.boxes is None:
        continue

    if tracking_result.boxes.id is None:
        continue

    boxes = tracking_result.boxes.xyxy.cpu().numpy()
    track_ids = tracking_result.boxes.id.int().cpu().numpy()

    # ==========================================
    # B. PROSES SETIAP BUAH
    # ==========================================

    if tracking_result.masks is None:
        continue

    # Ambil mask dan Track ID
    tracking_masks = tracking_result.masks.data.cpu().numpy()

    for track_mask, track_id in zip(tracking_masks, track_ids):

        # Sesuaikan ukuran mask dengan frame asli
        track_mask = cv2.resize(
            track_mask.astype(np.uint8),
            (width, height),
            interpolation=cv2.INTER_NEAREST
        )

        track_mask = (track_mask > 0).astype(np.uint8) * 255

        # Cari area terluar mask untuk menentukan ROI
        ys, xs = np.where(track_mask > 0)

        if len(xs) == 0 or len(ys) == 0:
            continue

        x1, x2 = xs.min(), xs.max() + 1
        y1, y2 = ys.min(), ys.max() + 1

        # Crop gambar dan tracking mask
        fruit_crop = image[y1:y2, x1:x2]
        track_mask_crop = track_mask[y1:y2, x1:x2]

        # Batasi gambar crop hanya pada area mask buah
        fruit_crop = cv2.bitwise_and(
            fruit_crop,
            fruit_crop,
            mask=track_mask_crop
        )

        # ======================================
        # C. SEGMENTASI BUAH
        # ======================================

        segmentation_results = segmentation_model.predict(
            source=fruit_crop,
            conf=SEG_CONF,
            retina_masks=True,
            verbose=False
        )

        seg_result = segmentation_results[0]

        if seg_result.masks is None:
            continue

        crop_height, crop_width = fruit_crop.shape[:2]

        predicted_masks = seg_result.masks.data.cpu().numpy()

        # Pilih mask dengan luas terbesar dalam crop
        fruit_mask_crop = max(
            predicted_masks,
            key=lambda mask: np.count_nonzero(mask > 0.5)
        )

        # Sesuaikan ukuran mask dengan ukuran crop
        fruit_mask_crop = cv2.resize(
            fruit_mask_crop.astype(np.uint8),
            (crop_width, crop_height),
            interpolation=cv2.INTER_NEAREST
        )

        fruit_mask_crop = (fruit_mask_crop > 0).astype(np.uint8) * 255

        fruit_mask_crop = cv2.bitwise_and(
            fruit_mask_crop,
            track_mask_crop
        )

        # Buat mask buah berukuran sama dengan frame asli
        fruit_mask = np.zeros(
            (height, width),
            dtype=np.uint8
        )

        fruit_mask[y1:y2, x1:x2] = fruit_mask_crop

        # ======================================
        # D. SEGMENTASI BERCAK
        # ======================================

        spot_mask = detect_spots(
            image,
            fruit_mask
        )

        # ======================================
        # E. HITUNG LUAS DAN PERSENTASE
        # ======================================

        fruit_area = np.count_nonzero(fruit_mask)
        spot_area = np.count_nonzero(spot_mask)

        if fruit_area == 0:
            continue

        spot_percentage = (
            spot_area / fruit_area
        ) * 100

        # Simpan hasil
        results_table.append({
            "Frame": os.path.basename(frame_path_str),
            "Track ID": int(track_id),
            "Fruit Area (px)": fruit_area,
            "Spot Area (px)": spot_area,
            "Spot Percentage (%)": round(spot_percentage, 2)
        })

        # Simpan data untuk visualisasi
        visualization_data.append({
            "frame": frame_index,
            "track_id": int(track_id),
            "image": image.copy(),
            "fruit_mask": fruit_mask.copy(),
            "spot_mask": spot_mask.copy(),
            "percentage": spot_percentage,
            "bbox": (x1, y1, x2, y2),
            "path": frame_path_str
        })

print("\nSemua frame selesai diproses!")

Processing frame 1/10
Processing frame 2/10
Processing frame 3/10
Processing frame 4/10
Processing frame 5/10
Processing frame 6/10
Processing frame 7/10
Processing frame 8/10
Processing frame 9/10
Processing frame 10/10

Semua frame selesai diproses!


In [40]:
df = pd.DataFrame(results_table)

display(df)

# Simpan tabel ke CSV
csv_path = os.path.join(
    OUTPUT_DIR,
    "guava_spot_percentage_tracking.csv"
)

df.to_csv(csv_path, index=False)

print(f"CSV berhasil disimpan di: {csv_path}")

""


CSV berhasil disimpan di: /content/drive/MyDrive/ggf_guava_project/output/guava_spot_percentage_tracking.csv


In [39]:
for frame_index, frame_path in enumerate(frame_paths):

    frame_data = [
        item for item in visualization_data
        if item["frame"] == frame_index
    ]

    if len(frame_data) == 0:
        continue

    print(f"\nFRAME: {os.path.basename(frame_path)}")

    # Satu baris untuk setiap buah
    fig, axes = plt.subplots(
        len(frame_data),
        3,
        figsize=(15, 5 * len(frame_data))
    )

    if len(frame_data) == 1:
        axes = np.array([axes])

    for row, item in enumerate(frame_data):

        image_rgb = cv2.cvtColor(
            item["image"],
            cv2.COLOR_BGR2RGB
        )

        x1, y1, x2, y2 = item["bbox"]

        # Gambar asli dengan bounding box
        image_box = image_rgb.copy()

        cv2.rectangle(
            image_box,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            image_box,
            f"ID: {item['track_id']}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Gambar asli
        axes[row, 0].imshow(image_box)
        axes[row, 0].set_title(
            f"Original - Track ID {item['track_id']}"
        )

        # Mask buah
        axes[row, 1].imshow(
            item["fruit_mask"],
            cmap="gray"
        )
        axes[row, 1].set_title("Fruit Mask")

        # Mask bercak
        axes[row, 2].imshow(
            item["spot_mask"],
            cmap="gray"
        )
        axes[row, 2].set_title(
            f"Spot Mask - {item['percentage']:.2f}%"
        )

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [41]:
def show_results(_=None):

    h_max = h_max_slider.value
    s_min = s_min_slider.value
    v_max = v_max_slider.value

    summary = []

    for i, data in enumerate(frame_data):

        image = data["image"]
        fruit_mask = data["fruit_mask"]

        # Konversi gambar ke HSV
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Threshold bercak
        lower = np.array([0, s_min, 0])
        upper = np.array([h_max, 255, v_max])

        spot_mask = cv2.inRange(hsv, lower, upper)

        # Batasi bercak hanya di dalam mask buah
        spot_mask = cv2.bitwise_and(
            spot_mask,
            fruit_mask
        )

        # Hitung luas dalam piksel
        fruit_area = np.count_nonzero(fruit_mask)
        spot_area = np.count_nonzero(spot_mask)

        # Persentase bercak terhadap luas buah
        percentage = (
            (spot_area / fruit_area) * 100
            if fruit_area > 0 else 0
        )

        summary.append({
            "Frame": os.path.basename(data["path"]),
            "Fruit Area (px)": fruit_area,
            "Spot Area (px)": spot_area,
            "Spot Percentage (%)": round(percentage, 2)
        })

        # Konversi gambar asli ke RGB untuk matplotlib
        original_rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # Tampilkan empat visualisasi
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        axes[0].imshow(original_rgb)
        axes[0].set_title("Original Image")

        axes[1].imshow(fruit_mask, cmap="gray")
        axes[1].set_title("Fruit Mask")

        axes[2].imshow(spot_mask, cmap="gray")
        axes[2].set_title("Spot Mask")

        # Overlay bercak pada gambar asli
        overlay = original_rgb.copy()
        overlay[spot_mask > 0] = [255, 0, 0]

        axes[3].imshow(overlay)
        axes[3].set_title(
            f"Spot Area: {percentage:.2f}%"
        )

        for ax in axes:
            ax.axis("off")

        plt.suptitle(
            f"Frame {i + 1}: {os.path.basename(data['path'])}",
            fontsize=14
        )

        plt.tight_layout()
        plt.show()

    # Tampilkan tabel hasil
    df = pd.DataFrame(summary)

    print("RINGKASAN HASIL")
    display(df)

    return df


# Tombol untuk menjalankan visualisasi
show_button = widgets.Button(
    description="Tampilkan hasil",
    button_style="success",
    icon="play"
)

output = widgets.interactive_output(
    lambda: None,
    {}
)

show_button.on_click(show_results)

display(show_button)

Button(button_style='success', description='Tampilkan hasil', icon='play', style=ButtonStyle())

RINGKASAN HASIL


""
